# Fase 06 - MCP Mermaid por Dominio (Processo Completo)

Este notebook documenta o fluxo completo para gerar, sincronizar e validar diagramas Mermaid por dominio no monorepo, com visao global da aplicacao e dependencias internas de cada modulo.

## 1) Objetivo

- Garantir 1 arquivo Mermaid por dominio em `apps/<dominio>/domain.mmd`.
- Incluir visao completa do projeto (todos os dominios).
- Incluir dependencias internas reais do modulo (controllers, providers, imports, injecao).
- Permitir sincronizacao automatica por container com `APP_NAME`.
- Expor um MCP HTTP na aplicacao para acionamento remoto da sincronizacao.

## 2) Componentes Implementados

1. Gerador central:
   - `tools/mcp/ensure-mermaid-domains.mjs`
2. Arquivos de saida por dominio:
   - `apps/<dominio>/domain.mmd`
3. Scripts npm:
   - `npm run mcp:mermaid:sync`
   - `npm run mcp:mermaid:sync:all`
4. MCP HTTP no app principal:
   - `GET /mcp/mermaid/status`
   - `POST /mcp/mermaid/sync`
   - `POST /mcp/mermaid/sync?domain=<dominio>`

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
required = [
    'tools/mcp/ensure-mermaid-domains.mjs',
    'apps/monorepo-ai-llm/src/mermaid-mcp.controller.ts',
    'apps/monorepo-ai-llm/src/mermaid-mcp.service.ts',
    'apps/monorepo-ai-llm/src/app.module.ts',
]

for rel in required:
    p = ROOT / rel
    print(('OK' if p.exists() else 'MISSING'), rel)

## 3) Fluxo de Geracao de Diagramas

O gerador faz:

1. Le os dominios em `apps/*`.
2. Analisa `src/*.ts` de cada dominio (exceto `*.spec.ts`).
3. Detecta classes exportadas e imports relativos/externos.
4. Extrai relacoes do modulo Nest (`imports`, `controllers`, `providers`, `exports`).
5. Extrai injecoes via construtor.
6. Monta `domain.mmd` com:
   - subgraph global de dominios
   - subgraph local do dominio
   - arestas de bootstrap/modulo/dependencias
   - dependencias externas detectadas

In [ ]:
import subprocess

cmd = ['node', 'tools/mcp/ensure-mermaid-domains.mjs']
res = subprocess.run(cmd, text=True, capture_output=True)
print('exit:', res.returncode)
print(res.stdout.strip() or '(sem stdout)')
if res.stderr.strip():
    print('stderr:')
    print(res.stderr.strip())

## 4) Sincronizacao por Dominio (modo container)

Use `APP_NAME` para o container sincronizar apenas o proprio dominio:

In [ ]:
import os, subprocess

env = dict(os.environ)
env['APP_NAME'] = 'sharepoint-api'
res = subprocess.run(['node', 'tools/mcp/ensure-mermaid-domains.mjs'], env=env, text=True, capture_output=True)
print('exit:', res.returncode)
print(res.stdout.strip() or '(sem stdout)')

## 5) MCP HTTP da Aplicacao

Endpoints disponiveis no app `monorepo-ai-llm`:

- `GET /mcp/mermaid/status`
- `POST /mcp/mermaid/sync`
- `POST /mcp/mermaid/sync?domain=users-api`

In [ ]:
import json, urllib.request

base = 'http://localhost:3000'
urls = [
    f'{base}/mcp/mermaid/status',
]

for url in urls:
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            data = r.read().decode('utf-8')
            print(url, '->', r.status)
            print(json.dumps(json.loads(data), indent=2, ensure_ascii=False))
    except Exception as e:
        print(url, '-> indisponivel:', e)

## 6) Exemplo de Integracao no Container

No entrypoint/startup do container:

```bash
export APP_NAME=users-api
npm run mcp:mermaid:sync
node dist/apps/users-api/main
```

Para sincronizar todos os dominios durante pipeline:

```bash
npm run mcp:mermaid:sync:all
```

## 7) Troubleshooting

- Erro `Script MCP nao encontrado`: execute no root do repo.
- Diagrama nao atualiza: valide `APP_NAME` e se existe `apps/<dominio>/src`.
- Endpoint indisponivel: subir `monorepo-ai-llm` e conferir porta `PORT`.
- CI: rode `npm run mcp:mermaid:sync:all` antes de empacotar artefatos.